# 10 · Case study — Copenhagen Municipality decision drafts (*KK afgørelse*)

A second real-world process modeled in Conductor: the Copenhagen
Municipality system that drafts official planning decisions for
*kolonihave* (allotment garden) building permits. Where the previous
notebook (`09`) showed wide parallel fan-out over chapters of an
environmental report, this one exercises a different pattern:

- **Parallel OCR** of N uploaded documents (for-each over files).
- A **long sequential chain** of 30+ rule-based checkpoints, each
  reading the running result of the previous ones.
- **Decision node + edge guards** that route to one of five draft
  generators by recommendation type (`byggetilladelse`, `mangelbrev`,
  `afslag`, `afvisning`, `viderebehandling`) — the canonical use of
  Conductor's CEL-on-edges branching.
- **Human-in-the-loop** approval gate: the case officer reviews the AI
  recommendation and may override the decision type before the draft
  is generated.
- **Conditional skip** via the SKIPPED sentinel: the matrikel WFS
  lookup is skipped when the data extractor couldn't find a matrikel
  number.

Stubs stand in for the real Mistral OCR, OpenAI, ChromaDB and
Copenhagen WFS clients so the notebook runs offline.


## 1 · Stub services


In [ ]:
from __future__ import annotations

import hashlib
import time
from typing import Annotated, Any, Literal


def fake_ocr(doc_id: str) -> dict[str, Any]:
    """Stand-in for Mistral OCR — returns a fake page+text per doc id."""
    fixtures = {
        "ansoegning.pdf": {
            "text": (
                "Ansøgning om byggetilladelse — Haveforening Solbakken, lod 42.\n"
                "Matrikel 12a, ejerlav 1234. Bebyggelsesprocent ansøgt 18%.\n"
                "Tilbygning til hus: 7 m² ekstra, samlet 56 m². Højde 3,1 m."
            ),
            "pages": 2,
        },
        "tegninger.pdf": {
            "text": (
                "Tegninger — plantegning + facader. Måltagning bekræftet.\n"
                "Afstand til skel: 2,6 m (krav 2,5 m)."
            ),
            "pages": 4,
        },
        "udtalelse.pdf": {
            "text": "Udtalelse fra haveforeningens bestyrelse: ingen indvendinger.",
            "pages": 1,
        },
    }
    return fixtures.get(doc_id, {"text": f"(no fixture for {doc_id})", "pages": 0})


def fake_llm(prompt: str, *, model: str = "gpt-stub") -> str:
    """Stand-in for OpenAI Chat Completions. Returns a deterministic
    fragment of realistic Danish prose, picked by the prompt's first line
    and the decision type if present."""
    head = prompt.splitlines()[0].lower() if prompt else ""
    if "byggetilladelse" in prompt.lower():
        return (
            "## Afgørelse\n\n"
            "Københavns Kommune meddeler hermed **byggetilladelse** til den ansøgte "
            "tilbygning på 7 m² på lod 42 i Haveforening Solbakken. Det samlede bebyggede "
            "areal udgør herefter 56 m², svarende til 18 % bebyggelsesprocent — under "
            "den tilladte grænse på 20 %.\n\n"
            "## Vilkår\n\n"
            "1. Byggeriet skal udføres i overensstemmelse med de godkendte tegninger "
            "dateret den 15. marts 2026.\n"
            "2. Afstanden til skel skal mindst være 2,5 m. Den projekterede afstand på "
            "2,6 m overholder kravet.\n"
            "3. Tilbygningens højde må ikke overstige 3,5 m. Projekteret højde: 3,1 m.\n\n"
            "## Klagevejledning\n\n"
            "Afgørelsen kan påklages til Planklagenævnet senest 4 uger fra modtagelsen."
        )
    if "afslag" in prompt.lower():
        return (
            "## Afgørelse\n\n"
            "Københavns Kommune meddeler hermed **afslag** på ansøgningen om tilbygning "
            "på lod 42 i Haveforening Solbakken.\n\n"
            "## Begrundelse\n\n"
            "Det ansøgte byggeri overskrider den tilladte bebyggelsesprocent på 20 % "
            "fastsat i kolonihaveforeningens lokalplan. Den ansøgte tilbygning ville "
            "bringe det samlede bebyggede areal op på 22 % — over grænsen.\n\n"
            "## Klagevejledning\n\n"
            "Afgørelsen kan påklages til Planklagenævnet senest 4 uger fra modtagelsen."
        )
    if "mangelbrev" in prompt.lower():
        return (
            "## Mangelbrev\n\n"
            "Til behandling af din ansøgning om byggetilladelse mangler vi følgende "
            "oplysninger:\n\n"
            "- Underskrevet udtalelse fra haveforeningens bestyrelse\n"
            "- Aktuelle tegninger med målfast plantegning\n"
            "- Dokumentation for afstand til naboskel\n\n"
            "Vi anmoder dig om at fremsende ovenstående **inden 14 dage** fra modtagelsen "
            "af dette brev. Modtages oplysningerne ikke rettidigt, vil sagen blive afsluttet "
            "uden afgørelse, jf. forvaltningslovens § 11."
        )
    if "afvisning" in prompt.lower():
        return (
            "## Afgørelse\n\n"
            "Københavns Kommune **afviser** ansøgningen, da det ansøgte byggeri ikke kan "
            "behandles efter byggeloven men kræver dispensation fra lokalplanen.\n\n"
            "## Vejledning\n\n"
            "Du henvises til at indsende en separat ansøgning om dispensation, hvorefter "
            "byggesagen kan genoptages."
        )
    if "escalate" in prompt.lower() or "videre" in prompt.lower():
        return (
            "## Foreløbig vurdering\n\n"
            "Sagen indeholder forhold der kræver yderligere sagsbehandling — herunder "
            "kontakt til haveforeningens bestyrelse og indhentelse af supplerende "
            "udtalelser. Sagen videresendes til specialiseret behandling."
        )
    if "summarize" in head or "summary" in head or "sammenfatning" in head:
        return (
            "Sagen vedrører en ansøgning om en 7 m² tilbygning på lod 42 i "
            "Haveforening Solbakken. Det samlede bebyggede areal udgør herefter "
            "56 m² (18 %), inden for tilladt grænse. Afstand til skel og højde "
            "overholder kravene. Ingen mangler observeret."
        )
    if "identify timeline" in head or "issues" in head:
        return (
            "Tidslinje: ansøgning indgivet 12. marts, tegninger fremsendt 15. marts, "
            "udtalelse fra bestyrelsen 18. marts. **Ingen modstridende oplysninger "
            "fundet** mellem dokumenterne."
        )
    digest = hashlib.sha1(prompt.encode()).hexdigest()[:6]
    return f"[{model}/{digest}] (no fixture) {head[:60]}"


def fake_wfs_matrikel(matrikelnummer: str, ejerlavskode: str) -> dict[str, Any]:
    return {
        "wfs_kontaminering": False,
        "wfs_fredet_bygning": False,
        "wfs_aa_beskyttelseslinje": True,
    }


def fake_lot_register(haveforening: str, lod: int) -> dict[str, Any]:
    # Pretend the official register says lod 42 is 392 m².
    return {"register_areal_m2": 392.0, "register_validation": {"matches": True}}


def fake_chroma(text: str, decision_type: str) -> dict[str, str] | None:
    return {"case_id": "PRIOR-2025-0099", "decision_text_preview": "..."}


## 2 · Domain types

`DecisionType` is the small string-enum that drives the routing in §6.
Five values; the engine picks among five draft-generator nodes via
edge guards.


In [ ]:
from pydantic import BaseModel
from conductor._sentinel import SKIPPED


DecisionType = Literal[
    "afvisning",        # rejected outright (wrong type or insufficient info)
    "afslag",           # refused on the merits
    "mangelbrev",       # request for missing info
    "viderebehandling", # escalate for further review
    "byggetilladelse",  # building permit granted
]


class CaseData(BaseModel):
    haveforening: str | None = None
    lod_nummer: int | None = None
    matrikelnummer: str | None = None
    ejerlavskode: str | None = None
    samlet_bebygget_areal_m2: float | None = None
    bebyggelsesprocent: float | None = None
    distance_to_skel_m: float | None = None
    height_m: float | None = None


class CheckpointResult(BaseModel):
    id: str
    status: Literal["pass", "fail", "skip"]
    reason: str = ""
    critical: bool = False


class CaseSummary(BaseModel):
    sagsinfo: dict[str, Any]
    vurdering: dict[str, Any]


# A trimmed checklist — the real system has 30+ across B (basic), R
# (regulation), W (WFS-derived), M (manual). We model 5 representative
# rules so the chain is visible without overwhelming the cell output.
CHECKLIST = ["B1_haveforening", "B2_lod", "R1_bebyggelse", "R2_skel", "W1_kontaminering"]


## 3 · Register the conductor nodes

Each LLM/agent call, each external integration, and each compound rule
becomes a node. The 5 chained checkpoints share state via Conductor's
`FlowStore` — a side-channel cache the engine auto-injects.


In [ ]:
import conductor_nodes
from conductor import GraphEdge, GraphNode, NodeRegistry, compile
from conductor.compound.for_each import FOR_EACH
from conductor.errors import FlowPausedException, HumanInputRequired
from conductor.execution.engine import collect, execute, resume
from conductor.execution.store import FlowStore
from conductor.widgets import Output, Text, Textarea

registry = NodeRegistry()
conductor_nodes.loop.register(registry)


@registry.node("ocr-document", version=1,
               name="OCR document",
               description="Mistral OCR per file (parallel inside for-each)",
               max_retries=2, retry_delay=0.1)
def ocr_document(
    doc_id: Annotated[str, Text(label="Doc id")],
) -> Annotated[dict, Output(label="OCR")]:
    time.sleep(0.05)  # simulate API latency so parallelism is visible
    return {"doc_id": doc_id, **fake_ocr(doc_id)}


@registry.node("combine-ocr", version=1,
               name="Combine OCR",
               description="Stitch per-doc OCR into one timeline string")
def combine_ocr(
    pages: Annotated[list[dict], Text(label="Pages")],
) -> Annotated[str, Output(label="Timeline")]:
    out: list[str] = []
    for p in pages:
        out.append(f"[=== Dokument: {p['doc_id']} ===]")
        out.append(p["text"])
    return "\n\n".join(out)


@registry.node("analyze-timeline", version=1,
               name="Analyze timeline",
               description="LLM reads the documents end-to-end and surfaces conflicts")
def analyze_timeline(
    timeline: Annotated[str, Textarea(label="Timeline")],
) -> Annotated[str, Output(label="Analysis")]:
    return fake_llm(f"Identify timeline issues:\n{timeline}", model="gpt-advanced")


@registry.node("extract-data", version=1,
               name="Extract case data",
               description="Structured-output LLM extracts ~60 fields")
def extract_data(
    timeline: Annotated[str, Textarea(label="Timeline")],
) -> Annotated[CaseData, Output(label="Case data")]:
    # The real extractor uses a pydantic JSON schema. Here we hard-code
    # what the deterministic fixtures contain.
    return CaseData(
        haveforening="Solbakken",
        lod_nummer=42,
        matrikelnummer="12a",
        ejerlavskode="1234",
        samlet_bebygget_areal_m2=56.0,
        bebyggelsesprocent=18.0,
        distance_to_skel_m=2.6,
        height_m=3.1,
    )


@registry.node("maybe-fetch-matrikel", version=1,
               name="Maybe fetch matrikel",
               description="Conditional WFS call — skipped if no matrikel number")
def maybe_fetch_matrikel(
    case_data: Annotated[CaseData, Text(label="Case data")],
) -> tuple[
    Annotated[Any, Output(label="Matrikel data (or skipped)")],
    Annotated[Any, Output(label="No-matrikel marker")],
]:
    case_data = CaseData.model_validate(case_data)
    if case_data.matrikelnummer and case_data.ejerlavskode:
        return fake_wfs_matrikel(case_data.matrikelnummer, case_data.ejerlavskode), SKIPPED
    return SKIPPED, "no matrikel — skipping WFS lookup"


@registry.node("validate-lot", version=1,
               name="Validate lot size",
               description="Pure Excel-register lookup")
def validate_lot(
    case_data: Annotated[CaseData, Text(label="Case data")],
) -> Annotated[dict, Output(label="Lot validation")]:
    case_data = CaseData.model_validate(case_data)
    if case_data.haveforening is None or case_data.lod_nummer is None:
        return {"register_validation": {"matches": False}}
    return fake_lot_register(case_data.haveforening, case_data.lod_nummer)


# The 5 sequential checkpoints. Each takes the previous results from
# FlowStore, evaluates one rule, and writes its own result back.
def _make_checkpoint(cp_id: str):
    @registry.node(f"checkpoint-{cp_id}", version=1,
                   name=f"Checkpoint {cp_id}",
                   description=f"Rule {cp_id}")
    def _cp(
        case_data: Annotated[CaseData, Text(label="Case data")],
        store: FlowStore,
    ) -> Annotated[CheckpointResult, Output(label="Result")]:
        prior: list[CheckpointResult] = list(store.get("checkpoints") or [])
        cd = CaseData.model_validate(case_data)
        result = _evaluate_rule(cp_id, cd, prior)
        prior.append(result)
        store.set("checkpoints", prior)
        return result
    _cp.__name__ = f"checkpoint_{cp_id}"
    return _cp


def _evaluate_rule(cp_id: str, cd: CaseData, prior: list) -> CheckpointResult:
    if cp_id == "B1_haveforening":
        ok = bool(cd.haveforening)
        return CheckpointResult(
            id=cp_id, status="pass" if ok else "fail",
            reason=(f"Haveforening identificeret: '{cd.haveforening}'" if ok
                    else "Kunne ikke identificere haveforeningen — kritisk for sagsbehandling"),
            critical=not ok,
        )
    if cp_id == "B2_lod":
        ok = cd.lod_nummer is not None
        return CheckpointResult(
            id=cp_id, status="pass" if ok else "fail",
            reason=(f"Lod nr. {cd.lod_nummer} fundet i ansøgningen" if ok
                    else "Lod-nummer mangler i ansøgningen"),
            critical=not ok,
        )
    if cp_id == "R1_bebyggelse":
        if cd.bebyggelsesprocent is None:
            return CheckpointResult(id=cp_id, status="fail",
                                    reason="Bebyggelsesprocent mangler i ansøgningen")
        ok = cd.bebyggelsesprocent <= 20.0
        return CheckpointResult(
            id=cp_id, status="pass" if ok else "fail",
            reason=(f"Bebyggelsesprocent {cd.bebyggelsesprocent} % — inden for "
                    f"tilladt grænse på 20 %" if ok
                    else f"Bebyggelsesprocent {cd.bebyggelsesprocent} % overskrider "
                    "tilladt grænse på 20 %"),
        )
    if cp_id == "R2_skel":
        if cd.distance_to_skel_m is None:
            return CheckpointResult(id=cp_id, status="fail",
                                    reason="Afstand til skel ikke angivet")
        ok = cd.distance_to_skel_m >= 2.5
        return CheckpointResult(
            id=cp_id, status="pass" if ok else "fail",
            reason=(f"Afstand til skel {cd.distance_to_skel_m} m — overholder kravet på 2,5 m"
                    if ok else
                    f"Afstand til skel {cd.distance_to_skel_m} m — under kravet på 2,5 m"),
        )
    if cp_id == "W1_kontaminering":
        if any(p["critical"] and p["status"] == "fail" for p in prior):
            return CheckpointResult(id=cp_id, status="skip",
                                    reason="Springet over — kritisk fejl på en tidligere kontrol")
        return CheckpointResult(id=cp_id, status="pass",
                                reason="WFS-opslag: ingen jordforurening, ingen fredning, "
                                "ingen Å-beskyttelseslinje konflikter")
    return CheckpointResult(id=cp_id, status="skip", reason="unknown rule")


for cp_id in CHECKLIST:
    _make_checkpoint(cp_id)


@registry.node("determine-recommendation", version=1,
               name="Determine recommendation",
               description="Priority logic over checkpoint results")
def determine_recommendation(
    cps: Annotated[list[CheckpointResult], Text(label="Checkpoints")],
) -> Annotated[DecisionType, Output(label="Recommendation")]:
    # Priority: afvisning > afslag > mangelbrev > viderebehandling > byggetilladelse.
    cps = [CheckpointResult.model_validate(c) for c in cps]
    if any(c.status == "fail" and c.critical for c in cps):
        return "afvisning"
    if any(c.status == "fail" for c in cps):
        return "afslag"
    if any(c.status == "skip" for c in cps):
        return "mangelbrev"
    return "byggetilladelse"


@registry.node("generate-summary", version=1,
               name="Generate case summary",
               description="LLM produces sagsinfo + vurdering for the UI; "
                           "stashes it on the FlowStore so the routed draft "
                           "generator can read it without an extra edge.")
def generate_summary(
    timeline: Annotated[str, Textarea(label="Timeline")],
    cps: Annotated[list[CheckpointResult], Text(label="Checkpoints")],
    store: FlowStore,
) -> Annotated[CaseSummary, Output(label="Summary")]:
    headlines = [c["reason"] or c["id"] for c in cps if c["status"] != "pass"]
    cps = [CheckpointResult.model_validate(c) for c in cps]
    n_pass = sum(1 for c in cps if c.status == "pass")
    n_fail = sum(1 for c in cps if c.status == "fail")
    summary = CaseSummary(
        sagsinfo={
            "haveforening": "Solbakken",
            "lod_nummer": 42,
            "matrikel": "12a",
            "ejerlavskode": "1234",
            "ansoegt_areal_m2": 56.0,
            "bebyggelsesprocent": 18.0,
        },
        vurdering={
            "hovedproblemer": headlines,
            "kontroller_bestaaet": f"{n_pass}/{len(cps)}",
            "kritiske_fejl": n_fail,
            "beslutning_anbefalet": "se anbefaling fra recommendation-noden",
        },
    )
    # FlowStore is conductor's side-channel cache. Putting the summary
    # here keeps it out of the decision-routed edge graph — every
    # draft-* node has *only* the routed `decision_type` as an input,
    # so when its incoming edge is skipped the node is correctly skipped.
    store.set("case_summary", summary.model_dump())
    return summary


@registry.node("review-decision", version=1,
               name="Review (human gate)",
               description="Case officer approves or overrides the recommendation",
               actor={"kind": "human", "role": "case_officer"},
               is_decision=True)
def review_decision(
    recommendation: Annotated[DecisionType, Text(label="AI recommendation")],
    summary: Annotated[CaseSummary, Text(label="Case summary")],
) -> Annotated[DecisionType, Output(label="Approved decision type")]:
    raise HumanInputRequired(
        prompt=f"AI recommends '{recommendation}'. Approve or override.",
        schema={"decision_type": "afvisning | afslag | mangelbrev | viderebehandling | byggetilladelse"},
    )


# Five draft generators — one per decision type. Each has only the
# routed `decision_type` as a graph input; the case summary is read
# out of FlowStore. That way, when the decision-edge-guard skips the
# incoming edge, the node has no live inputs and is auto-skipped.
def _make_draft_node(dtype: str):
    @registry.node(f"draft-{dtype}", version=1,
                   name=f"Draft: {dtype}",
                   description=f"LLM-generated decision letter for '{dtype}' cases")
    def _gen(
        decision_type: Annotated[DecisionType, Text(label="Decision type")],
        store: FlowStore,
    ) -> Annotated[str, Output(label="Markdown")]:
        summary = store.get("case_summary") or {}
        prompt = (
            f"Skriv afgørelse: {dtype}.\n\n"
            f"Sagsinfo: {summary.get('sagsinfo', {})}\n"
            f"Vurdering: {summary.get('vurdering', {})}"
        )
        return fake_llm(prompt, model="gpt-advanced")
    _gen.__name__ = f"draft_{dtype}"
    return _gen


for _dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling", "byggetilladelse"):
    _make_draft_node(_dt)


@registry.node("assemble-docx", version=1,
               name="Assemble docx",
               description="Pypandoc + ZIP surgery — splices markdown into the KK template")
def assemble_docx(
    draft: Annotated[str, Text(label="Active draft")],
) -> Annotated[dict, Output(label="Artifact")]:
    # Decision-edge-guards make sure exactly one of the 5 draft-* edges
    # is alive. The resolver collapses the single live edge into a scalar
    # value, so we receive the active draft directly.
    return {"filename": "afgoerelse.docx", "size_bytes": len(draft),
            "preview": draft[:160]}


print(f"Registered {len(registry.all())} nodes.")


## 4 · Layer 1 — OCR fan-out

`asyncio.gather(*[ocr_file(fp) for fp in files])` in the original
becomes a for-each region with `execution_mode="Parallel"`. Three fake
documents → three OCR calls in parallel → one combined timeline.


In [ ]:
ocr_nodes = [
    GraphNode("start", "for-each-start@1", {
        "items": ["ansoegning.pdf", "tegninger.pdf", "udtalelse.pdf"],
        "execution_mode": "Parallel",
    }),
    GraphNode("body", "ocr-document@1", None),
    GraphNode("end",  "for-each-end@1", None),
    GraphNode("combine", "combine-ocr@1", None),
]
ocr_edges = [
    GraphEdge("e1", "start", "body",    "output_1", "doc_id"),
    GraphEdge("e2", "body",  "end",     "result",   "item"),
    GraphEdge("e3", "end",   "combine", "result",   "pages"),
]

t0 = time.perf_counter()
ocr_compiled = compile(nodes=ocr_nodes, edges=ocr_edges,
                        registry=registry, compound_types=[FOR_EACH])
ocr_results = await collect(execute(ocr_compiled))
print(f"OCR'd 3 docs in {time.perf_counter() - t0:.2f}s")
print()
print(ocr_results["combine"]["result"][:200], "...")


## 5 · Layer 2 — sequential checkpoint chain

The original `KolonihavePipeline` runs ~30 rules in CHECKLIST order,
each receiving `results_so_far`. We model that as a chain of nodes
sharing state via `FlowStore` — explicit edges from one checkpoint to
the next would also work but obscure the rule logic.

```
  extract-data ──┐
                 ├─> cp B1 ──> cp B2 ──> cp R1 ──> cp R2 ──> cp W1 ──> recommendation
  fetch-matrikel ┘   (each writes its result into FlowStore.checkpoints)
```


In [ ]:
chain_nodes = [
    GraphNode("timeline", "combine-ocr@1", {"pages": [
        {"doc_id": d, **fake_ocr(d)}
        for d in ("ansoegning.pdf", "tegninger.pdf", "udtalelse.pdf")
    ]}),
    GraphNode("extract", "extract-data@1", None,
              produces={"result": "case data"}),
]

# 5 sequential checkpoints, each consuming case data via shared reference
# and depending on the previous via an edge.
prev = None
for i, cp_id in enumerate(CHECKLIST):
    nid = f"cp{i}"
    chain_nodes.append(
        GraphNode(nid, f"checkpoint-{cp_id}@1", None,
                  consumes={"case_data": ("extract", "result")})
    )

chain_edges = [
    GraphEdge("e0", "timeline", "extract", "result", "timeline"),
]
for i in range(1, len(CHECKLIST)):
    # Sequential ordering — each checkpoint depends on the previous.
    # `case_data` itself isn't ferried via this edge; it's a sequencing
    # dep so FlowStore writes happen in the right order. Wire to a
    # parameter that doesn't exist? No — wire the previous result through
    # a small hand-off using ConnectionList on `case_data` is overkill.
    # Cleanest: use the result handle on a dummy input — but Conductor
    # requires the target handle to be a real param. Solution: don't add
    # a sequencing edge; instead pre-collect all previous via a final
    # aggregator. (Sequential-via-FlowStore needs the engine to enforce
    # order; consumes-only deps + the engine's eager scheduler won't
    # serialise because the checkpoints are independent of each other.)
    pass

# Add an explicit chain by edging cp[i-1].result → cp[i] via a hidden
# param. We piggy-back on `case_data` — every cp accepts it, the
# resolver prefers edges over consumes, but the producer's output is
# also a CheckpointResult, not a CaseData. To preserve type safety,
# add an extra "after" parameter. To keep this notebook compact, we
# simply add a `dummy` ConnectionList input on a small helper that
# fans the chain.


> **Design note.** The Conductor scheduler runs independent nodes in
> parallel by default. A "sequential chain" means *artificial*
> ordering. The clean way to express it is to edge each step's output
> to the next step's input — but our checkpoints are heterogeneous
> (different rules, same return type) and don't really "consume" each
> other. So we collapse the chain into a single `run-checklist` node
> that iterates internally, writes per-rule results into `FlowStore`,
> and returns the list. The Conductor primitives at play here are
> *the node + FlowStore*, not parallelism — that's what the original
> pipeline needed too.


In [ ]:
@registry.node("run-checklist", version=1,
               name="Run checklist",
               description="Sequential 5-rule chain — writes through FlowStore")
def run_checklist(
    case_data: Annotated[CaseData, Text(label="Case data")],
    store: FlowStore,
) -> Annotated[list[CheckpointResult], Output(label="Checkpoints")]:
    cd = CaseData.model_validate(case_data)
    results: list[CheckpointResult] = []
    for cp_id in CHECKLIST:
        # Real evaluator would dispatch into the registered checkpoint-*
        # node by id; for the demo we call the helper directly.
        prior_dumped = [r.model_dump() for r in results]
        r = _evaluate_rule(cp_id, cd, prior_dumped)
        results.append(r)
        store.set(f"cp/{cp_id}", r.model_dump())
    return results


# Replace the chain with the single fold node.
chain_nodes = [
    GraphNode("timeline", "combine-ocr@1", {"pages": [
        {"doc_id": d, **fake_ocr(d)}
        for d in ("ansoegning.pdf", "tegninger.pdf", "udtalelse.pdf")
    ]}),
    GraphNode("extract", "extract-data@1", None),
    GraphNode("checks",  "run-checklist@1", None),
]
chain_edges = [
    GraphEdge("e0", "timeline", "extract", "result", "timeline"),
    GraphEdge("e1", "extract",  "checks",  "result", "case_data"),
]
chain_compiled = compile(nodes=chain_nodes, edges=chain_edges, registry=registry)
chain_results = await collect(execute(chain_compiled))

for cp in chain_results["checks"]["result"]:
    cp = CheckpointResult.model_validate(cp)
    print(f"  {cp.status:5s}  {cp.id:20s}  {cp.reason}")


## 6 · Layer 3 — decision-edge-guards for variant selection

Five draft generators, one decision node, one edge per branch. CEL
guards on each outgoing edge inspect the chosen `decision_type` and
make exactly one branch live; the other four are added to
`state.skipped_edges` and skip-propagate through any downstream
nodes. `assemble-docx` reads its `drafts` input via `ConnectionList`
— five edges in, exactly one with a real value, four SKIPPED.


In [ ]:
def build_routing_demo(*, override: str | None = None) -> tuple[list, list]:
    """Standalone graph: feed a recommendation through the human gate
    (we'll resume with `override` if given) and watch the decision node
    pick exactly one of five draft generators."""
    nodes = [
        GraphNode("rec",     "determine-recommendation@1", None),
        GraphNode("summary", "generate-summary@1", None),
        GraphNode("review",  "review-decision@1", None),
        GraphNode("assemble","assemble-docx@1", None),
    ]
    for dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling", "byggetilladelse"):
        nodes.append(GraphNode(f"d_{dt}", f"draft-{dt}@1", None))

    edges = [
        # The case-summary feeds every draft generator (broadcast).
        # Two ways: shared reference (one consume per node) or N edges.
        # We use the latter so the listing is explicit.
    ]
    # Decision branches. Each guard checks the approved decision_type.
    for dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling"):
        edges.append(GraphEdge(
            f"g_{dt}", "review", f"d_{dt}", "result", "decision_type",
            when=f"result == '{dt}'", priority=10,
        ))
    # The fifth (byggetilladelse) is the else fallback.
    edges.append(GraphEdge(
        "g_perm", "review", "d_byggetilladelse", "result", "decision_type",
    ))
    # Each draft generator → assemble. Summary lives in FlowStore.
    for dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling", "byggetilladelse"):
        edges.append(GraphEdge(f"a_{dt}", f"d_{dt}", "assemble", "result", "draft"))

    return nodes, edges


## 7 · Layer 4 — full pipeline with the human-in-the-loop gate

End-to-end: OCR → analyse → extract → matrikel (conditional) → lot →
checklist → recommendation → summary → **HITL pause** → routed draft
→ docx.


In [ ]:
def build_full_pipeline() -> tuple[list[GraphNode], list[GraphEdge]]:
    nodes: list[GraphNode] = [
        # OCR fan-out
        GraphNode("start", "for-each-start@1", {
            "items": ["ansoegning.pdf", "tegninger.pdf", "udtalelse.pdf"],
            "execution_mode": "Parallel",
        }),
        GraphNode("ocr",  "ocr-document@1", None),
        GraphNode("end",  "for-each-end@1", None),
        GraphNode("combine", "combine-ocr@1", None,
                  produces={"result": "timeline"}),
        # Linear analysis
        GraphNode("analysis", "analyze-timeline@1", None),
        GraphNode("extract",  "extract-data@1", None,
                  produces={"result": "case data"}),
        # Conditional matrikel WFS
        GraphNode("matrikel", "maybe-fetch-matrikel@1", None,
                  consumes={"case_data": ("extract", "result")}),
        GraphNode("lot",      "validate-lot@1", None,
                  consumes={"case_data": ("extract", "result")}),
        # Sequential checklist
        GraphNode("checks",   "run-checklist@1", None,
                  consumes={"case_data": ("extract", "result")}),
        # Recommendation + summary + human gate
        GraphNode("rec",      "determine-recommendation@1", None),
        GraphNode("summary",  "generate-summary@1", None),
        GraphNode("review",   "review-decision@1", None),
        # Five draft generators + assemble
        GraphNode("assemble", "assemble-docx@1", None),
    ]
    for dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling", "byggetilladelse"):
        nodes.append(GraphNode(f"d_{dt}", f"draft-{dt}@1", None))

    edges: list[GraphEdge] = [
        GraphEdge("e1", "start", "ocr",      "output_1", "doc_id"),
        GraphEdge("e2", "ocr",   "end",      "result",   "item"),
        GraphEdge("e3", "end",   "combine",  "result",   "pages"),
        GraphEdge("e4", "combine", "analysis", "result", "timeline"),
        GraphEdge("e5", "combine", "extract",  "result", "timeline"),
        GraphEdge("e6", "checks", "rec",      "result", "cps"),
        GraphEdge("e7", "checks", "summary",  "result", "cps"),
        GraphEdge("e8", "combine", "summary", "result", "timeline"),
        GraphEdge("e9", "rec",    "review",   "result", "recommendation"),
        GraphEdge("e10","summary","review",   "result", "summary"),
    ]
    # Decision-edge-guards: one guard per draft generator.
    for dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling"):
        edges.append(GraphEdge(
            f"g_{dt}", "review", f"d_{dt}", "result", "decision_type",
            when=f"result == '{dt}'", priority=10,
        ))
    edges.append(GraphEdge(
        "g_perm", "review", "d_byggetilladelse", "result", "decision_type",
    ))
    # Each draft → assemble. (Summary travels via FlowStore — see the node defs.)
    for dt in ("afvisning", "afslag", "mangelbrev", "viderebehandling", "byggetilladelse"):
        edges.append(GraphEdge(f"a_{dt}", f"d_{dt}", "assemble", "result", "draft"))

    return nodes, edges


full_nodes, full_edges = build_full_pipeline()
full_compiled = compile(
    nodes=full_nodes, edges=full_edges, registry=registry,
    compound_types=[FOR_EACH],
)
print(f"Pipeline: {len(full_nodes)} nodes, {len(full_edges)} edges, "
      f"{len(full_compiled.execution_order)} execution steps")
print(f"Type warnings: {len(full_compiled.type_warnings)}")

# Phase 1 — run until the human gate.
import json
try:
    await collect(execute(full_compiled))
    print("ERROR: should have paused")
except FlowPausedException as e:
    checkpoint = e.checkpoint
    print(f"\nPaused at: {checkpoint['waiting_node_id']}")
    print(f"Prompt:    {checkpoint['prompt']}")

# Phase 2 — case officer overrides the recommendation.
final = await collect(resume(full_compiled, checkpoint, response="byggetilladelse"))

# Inspect which draft branch fired.
fired = [k for k in final if k.startswith("d_")]
print(f"\nLive draft branches: {fired}")

artifact = final["assemble"]["result"]
print(f"\nArtifact: {artifact['filename']} ({artifact['size_bytes']} bytes)")
print(f"Preview:\n{artifact['preview']}")


## 8 · Streaming execution trace

What did each node actually do? `execute()` is an async generator —
we can listen to its events and build a per-node trace as the flow
runs. The table below shows every node's duration and a snippet of
its output, generated live during a fresh run.


In [ ]:
from IPython.display import Markdown, display


async def trace_run(compiled, *, response=None, checkpoint=None, max_chars=120):
    """Run the flow and return one row per node firing."""
    rows: list[dict[str, Any]] = []
    started: dict[str, float] = {}
    if checkpoint is not None:
        evs = resume(compiled, checkpoint, response=response)
    else:
        evs = execute(compiled)
    async for ev in evs:
        et = ev.get("type")
        nid = ev.get("node_id")
        if et == "node_start":
            started[nid] = time.perf_counter()
        elif et in ("node_complete", "node_error", "node_skipped"):
            t0 = started.pop(nid, None)
            ms = int((time.perf_counter() - t0) * 1000) if t0 else 0
            res = ev.get("result")
            snippet = ""
            if et == "node_complete" and isinstance(res, dict):
                v = res.get("result", res)
                snippet = repr(v) if not isinstance(v, str) else v.replace("\n", " ")
                snippet = snippet[:max_chars] + ("…" if len(snippet) > max_chars else "")
            elif et == "node_error":
                snippet = str(ev.get("error", ""))[:max_chars]
            rows.append({"id": nid, "status": et, "ms": ms, "out": snippet})
        elif et == "flow_paused":
            return rows, ev["checkpoint"]
        elif et in ("flow_error", "flow_cancelled", "flow_timeout"):
            break
    return rows, None


def render_trace(rows, compiled):
    icon = {"node_complete": "✅", "node_error": "❌", "node_skipped": "⏭️"}
    lines = ["| # | node | type | ms | output |", "|---|---|---|---|---|"]
    for i, r in enumerate(rows, 1):
        node = compiled.node_map.get(r["id"])
        ntype = node.type.split("@")[0] if node else r["id"]
        ic = icon.get(r["status"], r["status"])
        out = r["out"].replace("|", "\\|").replace("\n", " ")
        lines.append(f"| {i} | `{r['id']}` | `{ntype}` | {r['ms']} | {ic} {out} |")
    return "\n".join(lines)


# Build a fresh pipeline so we can pause + resume with the trace.
trace_nodes, trace_edges = build_full_pipeline()
trace_compiled = compile(
    nodes=trace_nodes, edges=trace_edges,
    registry=registry, compound_types=[FOR_EACH],
)

# Phase 1: run until the human gate, recording each node.
rows1, ckpt = await trace_run(trace_compiled)
print("=== Phase 1 — automated steps (paused at human gate) ===")
display(Markdown(render_trace(rows1, trace_compiled)))

# Phase 2: resume with the case officer's decision.
rows2, _ = await trace_run(trace_compiled, response="byggetilladelse", checkpoint=ckpt)
print("\n=== Phase 2 — after human approval ===")
display(Markdown(render_trace(rows2, trace_compiled)))


## 9 · Drill-down — what's actually in the final draft?

The decision letter the customer would receive. With our fixtures all
checks pass, so the AI recommends *byggetilladelse* and the case
officer's approval routes the flow to `draft-byggetilladelse`. The
artifact's `preview` field above is truncated to 160 chars; the
actual draft is several paragraphs of Danish municipal prose.


In [ ]:
import json as _json

# Re-run end-to-end and resume with the same recommendation so we can
# inspect every per-node output without re-running by hand.
rows_full, ckpt_full = await trace_run(trace_compiled)
final_results = await collect(resume(trace_compiled, ckpt_full, response="byggetilladelse"))

# Each registered node's output, post-pipeline.
print("=== OCR'd timeline (combined) ===")
print(final_results["combine"]["result"][:400], "…\n")

print("=== Extracted case data ===")
print(_json.dumps(final_results["extract"]["result"], indent=2, ensure_ascii=False), "\n")

print("=== Checkpoint chain ===")
for cp in final_results["checks"]["result"]:
    cp = CheckpointResult.model_validate(cp)
    icon = {"pass": "✅", "fail": "❌", "skip": "⏭️"}[cp.status]
    print(f"  {icon}  {cp.id:24s}  {cp.reason}")

print(f"\n=== AI recommendation: {final_results['rec']['result']} ===")
print(f"=== Officer's choice:   {final_results['review']['result']} ===\n")

print("=== Generated decision letter ===")
print(final_results["assemble"]["result"]["preview"])
print()
print("(showing the first 160 chars — the full text:)\n")
# The live draft branch keeps its full output in state.results.
for nid, res in final_results.items():
    if nid.startswith("d_") and isinstance(res, dict) and "result" in res:
        print(res["result"])


## 10 · Variation — accept the AI recommendation

Same flow, but the human accepts whatever the AI suggested. With our
fixtures every checkpoint passes, so the recommendation is
`byggetilladelse` and the same branch fires. Try editing the fixtures
to see a different decision type travel through the gate.


In [ ]:
final2 = await collect(resume(full_compiled, checkpoint, response="afslag"))
fired2 = [k for k in final2 if k.startswith("d_")]
print(f"Live draft branch (override = afslag): {fired2}")
print(f"Artifact preview: {final2['assemble']['result']['preview']}")


## 11 · Visualizing the routed graph

Five draft-generators, one human gate, one decision routing — easier
to read as a Mermaid diagram. Decision diamonds, dashed
shared-reference arrows, and CEL guards rendered between ⟦ ⟧.


In [ ]:
from conductor.viz import to_mermaid
from IPython.display import Markdown

Markdown("```mermaid\n" + to_mermaid(full_compiled, direction="LR") + "\n```")


## 12 · What we modeled

| KK module / function | Conductor primitive |
|---|---|
| `app/api/helpers/document_processor.py:ocr_pdf` (Mistral OCR) | `ocr-document` node body inside a `for-each` over the file list |
| `app/api/prompt_funcs.py:analyze_documents_timeline` | `analyze-timeline` node |
| `app/api/prompt_funcs.py:extract_kolonihave_data` | `extract-data` node returning `CaseData` |
| `app/api/helpers/wfs_matrikel.py:fetch_matrikel_data` (conditional) | `maybe-fetch-matrikel` node — SKIPPED-sentinel branching when matrikel is missing |
| `app/api/helpers/lot_size_register.py:validate_lodstorrelse` | `validate-lot` node — pure Excel lookup |
| `app/api/checkpoints/checkpoints_kolonihave.py` (~30 sequential rules) | `run-checklist` fold over a static `CHECKLIST`, writes per-rule results to `FlowStore` |
| `KolonihavePipeline._get_recommendation` | `determine-recommendation` node (priority logic) |
| `app/api/decision_generator.py:create_case_summary` | `generate-summary` node |
| Frontend review step (case officer overrides decision type) | `review-decision` node raising `HumanInputRequired` |
| `generate_tilladelse_draft` / `generate_afslag_draft` / 4 others | 5 `draft-*` nodes, each gated by a CEL guard `result == '<type>'` on its incoming edge |
| `app/api/draft_md_to_docx.py:draft_to_docx` | `assemble-docx` node, reads draft markdown via `ConnectionList` (4 SKIPPED + 1 live → resolver picks the live one) |
| `chroma.find_similar_cases` (similar-case retrieval) | One more IO node — omitted here for brevity |

Things deliberately out of scope of the example:

- The 30+ rule **catalog** of checkpoints (we model 5).
- The semantic search step against ChromaDB. Same shape as the WFS
  call — single IO node, conditional on having a non-empty summary.
- The OOXML surgery in `draft_to_docx`. We return a synthetic dict
  artifact; the structural primitives are unchanged.
- The **checkpoint sub-flow**. The original lets later checkpoints
  branch on earlier ones; we collapse that into a single fold node
  + `FlowStore` writes. A more faithful model would register each
  checkpoint as its own node and chain them with edges or use a
  `while` loop. Pick whichever matches your editor/UI.
